# 主成分分析

## 课程：金融数据分析与建模

金融数据中充满了「信息冗余」：股票收益率之间高度相关，宏观指标往往同涨同跌，风险因子之间存在系统性的共变关系。面对几十甚至上百个变量时，直接建模不仅计算上低效，还容易陷入多重共线性的困境。

**主成分分析**（Principal Component Analysis，PCA）是处理这类问题的基础工具。它的核心思想是：将原始的 $p$ 个相关变量，转换为 $k \ll p$ 个互不相关的**主成分**（principal components），同时尽可能保留原始数据中的信息。

本章沿以下主线展开：

$$
\text{方差最大化的直觉}
\longrightarrow
\text{载荷与得分}
\longrightarrow
\text{选择主成分数量}
\longrightarrow
\text{双标图与解读}
\longrightarrow
\text{主成分回归}
$$

---

**本章使用的数据集**

| 数据集 | 变量 | 说明 |
|---|---|---|
| 数据集 S | 6 支股票月度收益率（200 个月） | 含两个潜在因子：市场因子和行业因子 |
| 数据集 M | 8 个宏观经济指标（100 个季度）+ GDP 增速 | 变量间高度共线，用于展示主成分回归 |

两个数据集均为模拟数据，设计上具有清晰的因子结构，便于验证 PCA 的提取结果。

---

## 从一个金融问题出发

考虑持有 6 支股票的投资组合：3 支科技股（TECH1–3）和 3 支金融股（FIN1–3）。每个月，6 支股票的收益率以不同幅度涨跌，但它们并非完全独立——在「牛市」中倾向于集体上涨，在「熊市」中集体下跌。

图 1 展示了这 6 支股票以 TECH1 为基准的散点图。

![股票收益散点图](./figs/ml_PCA_fig01_raw_scatter.png){width=100%}

**图 1** 科技股之间（蓝色点）和金融股之间（黄色点）相关性很高——点沿斜线分布。但科技股与金融股之间（蓝黄混合区域）的关系更复杂：有正相关，但斜率不同，说明还有其他维度的信息。

面对这样的数据，一个自然的问题是：

> 这 6 个变量背后，究竟有几个独立的驱动力？

直觉告诉我们：「所有股票一起涨跌」是一个驱动力（市场整体走势），「科技股涨而金融股跌，或反之」是另一个驱动力（行业轮动）。PCA 就是将这种直觉数量化的工具——它把 6 维数据压缩为少数几个主成分，每个主成分对应一种独立的变动模式。

---

## 方差最大化：PCA 的核心思想

### 投影与方差

PCA 的出发点是一个几何问题：在所有可能的方向中，找一个方向 $\mathbf{v}$，使样本在这个方向上的**投影方差最大**。

设 $\mathbf{x}_i \in \mathbb{R}^p$ 是第 $i$ 个样本（已中心化），在方向 $\mathbf{v}$（单位向量）上的投影坐标为

$$
z_i = \mathbf{v}^\top \mathbf{x}_i \tag{1}
$$

所有样本在这个方向上的投影方差为

$$
\text{Var}(z) = \frac{1}{n} \sum_{i=1}^n z_i^2
= \frac{1}{n} \sum_{i=1}^n (\mathbf{v}^\top \mathbf{x}_i)^2
= \mathbf{v}^\top \mathbf{S} \mathbf{v} \tag{2}
$$

其中 $\mathbf{S} = \frac{1}{n} \mathbf{X}^\top \mathbf{X}$ 是样本协方差矩阵。

**第一主成分**就是使公式 (2) 最大的方向，约束为 $\|\mathbf{v}\|=1$：

$$
\mathbf{v}_1 = \arg\max_{\|\mathbf{v}\|=1} \mathbf{v}^\top \mathbf{S} \mathbf{v} \tag{3}
$$

可以证明，这个问题的解恰好是 $\mathbf{S}$ 的**最大特征值**对应的**特征向量**。（完整推导见附录 A。）

图 2 直观展示了方差最大化的含义：选择不同的投影方向，样本点散布的程度（方差）差异显著。

![方差最大化方向](./figs/ml_PCA_fig02_variance_direction.png){width=100%}

**图 2** 两个子图使用同一份数据（TECH1 和 FIN1 的标准化收益率）。左图：沿水平轴（任意方向）的投影，方差较小；右图：沿第一主成分方向的投影，方差明显更大。橙色/蓝色小点是各样本在该方向上的投影点，灰色细线是投影的「垂足连线」。

### 后续主成分：正交约束下的方差最大化

**第二主成分** $\mathbf{v}_2$ 在与 $\mathbf{v}_1$ **正交**的约束下，再次最大化投影方差：

$$
\mathbf{v}_2 = \arg\max_{\|\mathbf{v}\|=1,\, \mathbf{v}^\top \mathbf{v}_1 = 0}
\mathbf{v}^\top \mathbf{S} \mathbf{v} \tag{4}
$$

正交约束保证了第二主成分携带的信息与第一主成分**不重叠**。依此类推，第 $k$ 个主成分在与前 $k-1$ 个主成分都正交的约束下最大化方差。

最终得到 $p$ 个主成分方向，它们恰好是 $\mathbf{S}$ 的 $p$ 个特征向量，按特征值从大到小排列：$\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_p \geq 0$。

---

## PCA 即坐标系旋转

PCA 还有一个等价的几何解释：**将原始坐标系旋转（正交变换）到一个新的坐标系**，使数据在新坐标轴上的方差尽可能集中在前几个轴上，且各轴之间互不相关。

图 3 展示了这一旋转过程。

![坐标系旋转](./figs/ml_PCA_fig03_rotation.png){width=100%}

**图 3** (a) 原始标准化空间中，两个主成分方向用箭头标出；(b) 旋转到主成分坐标系后，数据的最大变动方向与 PC1 轴对齐，PC1 和 PC2 轴互相垂直，且 PC1 方向的散布明显比 PC2 宽——即 PC1 的方差显著大于 PC2。

::: {.callout-note}
### PCA 与坐标旋转的代数关系

设 $\mathbf{V} = [\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_p]$ 为主成分方向矩阵（列为特征向量），则主成分变换可以写成

$$
\mathbf{Z} = \mathbf{X} \mathbf{V} \tag{5}
$$

其中 $\mathbf{X}$ 是 $n \times p$ 的（已中心化）数据矩阵，$\mathbf{Z}$ 是 $n \times p$ 的主成分得分矩阵。

由于 $\mathbf{V}$ 是正交矩阵（$\mathbf{V}^\top \mathbf{V} = \mathbf{I}$），变换是一个**保距旋转**：样本之间的欧氏距离在变换前后完全保持不变。这意味着 PCA 不会丢失任何信息——只是换了一个坐标系来看同一份数据。
:::

---

## 三个核心概念：载荷、得分、解释方差

在正式对 6 维股票数据做 PCA 之前，先把三个最核心的术语定义清楚。

### 主成分载荷（Loadings）

第 $k$ 个主成分的**载荷向量** $\mathbf{v}_k \in \mathbb{R}^p$ 是协方差矩阵 $\mathbf{S}$ 第 $k$ 大特征值对应的特征向量。载荷的第 $j$ 个分量 $v_{jk}$ 表示：

> 原始变量 $x_j$ 在第 $k$ 个主成分上的权重（贡献）。

$|v_{jk}|$ 越大，说明 $x_j$ 对第 $k$ 个主成分的贡献越大。

### 主成分得分（Scores）

第 $i$ 个样本在第 $k$ 个主成分上的**得分**是该样本沿 $\mathbf{v}_k$ 方向的投影：

$$
z_{ik} = \mathbf{v}_k^\top \mathbf{x}_i = \sum_{j=1}^p v_{jk} x_{ij} \tag{6}
$$

得分矩阵 $\mathbf{Z} \in \mathbb{R}^{n \times p}$ 的每一列代表所有样本在该主成分上的分布；不同列之间的样本相关系数为零（正交性保证）。

### 解释方差比（Explained Variance Ratio）

第 $k$ 个主成分能解释的方差占总方差的比例为

$$
\text{EVR}_k = \frac{\lambda_k}{\sum_{j=1}^p \lambda_j} \tag{7}
$$

其中 $\lambda_k$ 是 $\mathbf{S}$ 的第 $k$ 大特征值。这个比例直接告诉我们：保留前 $k$ 个主成分，能保留原始数据多少比例的「方差信息」。

---

## 碎石图：如何选择主成分数量

对 6 维股票收益数据做 PCA 后，首先看**碎石图**（Scree Plot）——它展示每个主成分的解释方差比，以及累计解释方差。

![碎石图](./figs/ml_PCA_fig04_screeplot.png){width=100%}

**图 4** 数据集 S 的碎石图。柱状表示各主成分的解释方差，折线表示累计解释方差。从图中可以看到一个明显的「肘部」（elbow）：PC1 和 PC2 各自解释了大部分方差，PC3 之后的方差贡献迅速减小且趋于平稳。

碎石图说明：**保留前 2 个主成分已经足够**。这与数据的生成机制完全一致——数据集 S 本来就是由两个潜在因子（市场因子和行业因子）驱动的，PCA 成功地把这两个因子「还原」出来了。

### 选择主成分数量的常用规则

| 规则 | 做法 | 适用场景 |
|---|---|---|
| **肘部法则** | 在碎石图中找方差曲线斜率急剧变小的拐点 | 最常用，直观 |
| **累计方差阈值** | 保留使累计方差超过 80%（或 90%）的最少主成分 | 降维/压缩场景 |
| **Kaiser 准则** | 保留特征值 $\lambda_k > 1$ 的主成分（标准化数据） | 探索性因子分析 |
| **交叉验证** | 在下游任务（如回归）中用 CV 选最优 $k$ | 以预测为目标时 |

没有哪个规则是绝对正确的。实践中常常结合碎石图和具体的解释目的来决定。

::: {.callout-note}
### 标准化的重要性

PCA 对变量的量纲非常敏感。如果直接对原始数据做 PCA，量纲大的变量（如用「元」衡量的股价）会主导第一主成分，而量纲小的变量（如用「%」衡量的收益率）几乎被忽略。

**几乎所有情况下，应先对数据做标准化（StandardScaler），再做 PCA。**

标准化等价于对**相关矩阵**（而非协方差矩阵）做特征分解。标准化后每个变量的方差为 1，总方差等于变量数 $p$，每个主成分的解释方差比可以直接理解为「解释了 $p$ 份方差中的几份」。
:::

---

## 解读主成分：载荷热力图

碎石图告诉我们「要几个主成分」，载荷矩阵告诉我们「每个主成分的经济含义」。

![载荷热力图](./figs/ml_PCA_fig05_loadings.png){width=80%}

**图 5** 前两个主成分的载荷热力图。红色代表正载荷，蓝色代表负载荷，颜色深浅代表绝对值大小。

从图 5 可以读出以下信息：

**PC1（第一主成分）**：所有 6 支股票的载荷符号相同，均为正。这意味着 PC1 得分高时，所有股票同时上涨；PC1 得分低时，所有股票同时下跌。这是典型的**市场因子（Market Factor）**——对应整个股市的系统性涨跌。

**PC2（第二主成分）**：科技股（TECH1–3）载荷为正，金融股（FIN1–3）载荷为负。这意味着 PC2 得分高时，科技股上涨而金融股下跌；PC2 得分低时，反之。这是典型的**行业因子（Sector Factor）**——对应科技与金融板块的轮动。

**结论**：PCA 在不知道任何先验金融知识的情况下，自动从数据中提取出了与金融理论高度吻合的两个因子。这正是 PCA 在量化金融领域被广泛使用的原因之一。

::: {.callout-important}
### 主成分的方向符号没有经济含义

PCA 的解只在方向上唯一——对每个特征向量 $\mathbf{v}_k$，$-\mathbf{v}_k$ 同样是合法的解（它指向相反方向，但解释同等数量的方差）。

这意味着：如果某次计算发现 PC1 所有载荷为负，只需乘以 $-1$ 翻转方向，意义完全不变。在解读载荷时，应关注的是**各变量之间载荷的相对大小和符号关系**，而不是载荷的绝对正负。
:::

---

## 双标图：同时展示样本和变量

**双标图**（Biplot）是 PCA 最重要的可视化工具，它将**样本得分**（散点）和**变量载荷**（箭头）叠加在同一张图上，可以同时回答两个问题：哪些样本相似？哪些变量相关？

![双标图](./figs/ml_PCA_fig06_biplot.png){width=90%}

**图 6** 股票收益数据集的双标图。散点是 200 个月的样本（颜色与强弱有关）；带标签的箭头代表 6 支股票，箭头方向和长度反映该股票在主成分空间中的载荷。

**双标图的解读规则：**

- **箭头方向相近**（夹角小）的两个变量，在原始空间中正相关。  图 6 中 TECH1、TECH2、TECH3 的箭头几乎平行，说明三支科技股高度正相关；  FIN1、FIN2、FIN3 同理。

- **箭头方向相反**（夹角接近 180°）的变量，在原始空间中负相关。  科技股箭头（朝右上）和金融股箭头（朝右下）的 PC2 分量方向相反，  说明在行业因子维度上两组股票反向运动。

- **箭头垂直**（夹角约 90°）的变量，近似不相关。

- **箭头长度**反映该变量被前两个主成分解释的程度：箭头越长，  说明该变量的大部分方差被这两个主成分捕捉到了。

- **样本点的位置**：沿 PC1 方向（横轴）偏右的月份是「牛市月份」（整体上涨），  偏左的是「熊市月份」；沿 PC2 方向（纵轴）偏上的月份是「科技强势月份」。

---

## 主成分得分：样本在新坐标系中的坐标

主成分**得分**是每个样本在主成分坐标系中的位置，是 PCA 的核心输出之一，可以直接用于后续分析（降维、回归、聚类等）。

![主成分得分散点图](./figs/ml_PCA_fig07_scores_scatter.png){width=100%}

**图 7** (a) 200 个月在 PC1-PC2 平面上的散点，按 PC1 得分高低着色：黄色点（PC1 高分）对应市场强势月份，红色点（PC1 低分）对应市场弱势月份；(b) PC1 得分的时序图，其波动形态与「市场综合指数」高度相似——这验证了 PC1 确实在捕捉市场整体状态。

主成分得分有几个重要性质：

- **不同主成分得分之间的相关系数为零**（正交性的直接推论）
- **每个主成分得分列的方差等于对应特征值** $\lambda_k$
- **得分矩阵保留了原始数据的完整信息**（当保留所有 $p$ 个主成分时）

这最后一条说明 PCA 是**无损变换**——用前 $k$ 个主成分只是选择性地丢弃了方差较小的维度（信息压缩），而不是随意截断。

---

## 重构误差：丢失了多少信息？

用前 $k$ 个主成分重构原始数据的公式为：

$$
\hat{\mathbf{X}}^{(k)} = \mathbf{Z}_k \mathbf{V}_k^\top
= \sum_{j=1}^k z_{\cdot j} \mathbf{v}_j^\top \tag{8}
$$

其中 $\mathbf{Z}_k$ 是取前 $k$ 列的得分矩阵，$\mathbf{V}_k$ 是取前 $k$ 列的载荷矩阵。

重构误差（MSE）等于被丢弃的主成分所携带的方差之和：

$$
\text{MSE}^{(k)} = \frac{1}{np} \|\mathbf{X} - \hat{\mathbf{X}}^{(k)}\|_F^2
= \frac{1}{p} \sum_{j=k+1}^p \lambda_j \tag{9}
$$

图 8 用散点图展示了不同 $k$ 值下重构值与真实值的吻合程度。

![重构误差](./figs/ml_PCA_fig08_reconstruction.png){width=100%}

**图 8** 不同主成分数量的重构效果（以 TECH1 为例）。横轴为原始标准化值，纵轴为重构值，虚线为完美重构参考线（$y=x$）。$k=1$ 时偏差明显；$k=2$ 时已相当接近；$k=6$ 时完美重构（MSE=0），散点全部落在参考线上。

---

## PCA 与因子分析



::: {.callout-note}
### PCA 与因子分析（FA）的联系与区别

PCA 和因子分析（Factor Analysis，FA）都是降维方法，在金融文献中经常被混用，但它们的出发点不同：

| | PCA | 因子分析（FA） |
|---|---|---|
| **目标** | 最大化方差解释 | 建模变量的协方差结构 |
| **假设** | 无参数假设，纯数学分解 | 有明确的概率模型（潜在因子 + 特质误差） |
| **分解对象** | 总方差（含特质方差） | 共同方差（排除特质方差） |
| **旋转** | 主成分方向唯一确定 | 因子载荷可旋转（如 Varimax），提升可解释性 |
| **结果解读** | 主成分是原始变量的线性组合 | 因子是驱动变量共同变化的潜在力量 |

**在什么情况下用哪个？**

- 目标是**降维和数据压缩**（减少变量数、缓解多重共线性）→ PCA
- 目标是**理解变量背后的潜在结构**（如风险因子、消费偏好）→ FA
- 样本量较小、变量间共同方差比例较高 → FA 更合适
- 需要快速、无假设的线性变换 → PCA 更合适

在量化金融的实践中，两者都被广泛使用，PCA 更多出现在统计处理流程中，FA 更多出现在风险模型的结构设定中。
:::

---

## 主成分数量的选择：以宏观数据为例

现在用数据集 M（8 个宏观指标）进一步展示如何选择主成分数量。这个数据集的特点是：8 个变量背后只有 3 个真实的独立驱动力，因此变量间存在严重的多重共线性。

![宏观数据碎石图](./figs/ml_PCA_fig09_cumvar.png){width=100%}

**图 9** 宏观指标数据集的碎石图。前 3 个主成分已累计解释超过 85% 的方差；PC4 之后的贡献都很小，且累计方差曲线趋于平稳。两条参考线分别标出 80% 和 90% 的阈值。

肘部出现在 PC3 附近，与数据生成时预设的「3 个独立驱动因子」完全吻合。这说明即使在不知道数据真实结构的情况下，碎石图也能给出准确的信号。

---

## 主成分回归（PCR）

### 问题背景：多重共线性

当解释变量之间高度相关时，OLS 回归会遇到**多重共线性**问题：

- 回归系数的标准误膨胀，估计不稳定
- 不同的随机种子会得到截然不同的系数
- 模型在训练集表现好，但在新数据上泛化差

数据集 M 中，PMI、工业产值增速和零售增速三个变量之间的相关系数超过 0.85，CPI 和 PPI 的相关系数超过 0.90——这正是 OLS 不稳定的典型情形。

### PCR 的思路

**主成分回归**（Principal Component Regression，PCR）用以下两步解决多重共线性：

1. 对 $p$ 个解释变量做 PCA，取前 $k$ 个主成分得分 $\mathbf{Z}_k$（各列互不相关）
2. 以 $\mathbf{Z}_k$ 为解释变量做 OLS 回归

由于主成分得分之间不相关，步骤 2 中的 OLS 不再受多重共线性困扰。参数 $k$ 用交叉验证选择。

![PCR vs OLS](./figs/ml_PCA_fig10_pcr_vs_ols.png){width=100%}

**图 10** (a) PCR 在不同主成分数量下的 5 折 CV MSE，以及 OLS 全变量的基准（橙色虚线）。PCR 在 $k$ 较小时 CV 误差低于 OLS，说明降维不仅减少计算量，也改善了泛化性能；当 $k=p=8$ 时，PCR 退化为 OLS（误差相同）。(b) 最优 $k$ 对应的 PCR 预测值 vs 真实 GDP 增速。

### PCR 与 Ridge 回归的关系

PCR 和 Ridge 回归都是解决多重共线性的工具，思路不同：

- **PCR**：直接丢弃方差小的主成分方向，对保留方向的系数做普通 OLS
- **Ridge**：保留所有方向，但对方差小的方向对应的系数施加更大的收缩

两者都可以写成对主成分的操作：在主成分坐标系中，PCR 对第 $k$ 个方向的收缩因子是 $\mathbf{1}(\lambda_k > $ 阈值$)$（硬截断），而 Ridge 的收缩因子是 $\lambda_k / (\lambda_k + \eta)$（软收缩）。

因此，Ridge 通常被认为是 PCR 的「软版本」，在实践中往往具有更好的连续性。

::: {.callout-tip}
### 标准化在 PCR 中尤为重要

在 PCR 的完整流程中，标准化必须在交叉验证的**每个 fold 内部**独立执行，而不是在分割训练/验证集之前统一执行。

原因：如果先对全部数据标准化再划分 fold，验证集的均值和标准差信息就已经「泄漏」到了训练过程中，导致 CV 误差被低估。

使用 sklearn 的 `Pipeline` 可以自动避免这个问题：

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),   # 在每个 fold 内独立 fit
    ('pca',    PCA(n_components=3)),
    ('reg',    LinearRegression()),
])
# 直接传入原始数据，Pipeline 内部处理标准化
scores = cross_val_score(pipe, X_M, y_M, cv=5,
                          scoring='neg_mean_squared_error')
```
:::

---

## Python 实践

### 数据准备与标准化

```python
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 读取数据（由 ml_PCA_codes.ipynb 生成）
df = pd.read_csv('./data/pca_stock_returns.csv')
X = df.values          # (200, 6)
stock_names = df.columns.tolist()

# 标准化：必须在 fit 之前完成
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

print(f'均值（应接近 0）: {X_sc.mean(axis=0).round(6)}')
print(f'标准差（应接近 1）: {X_sc.std(axis=0).round(6)}')
```

### 拟合 PCA 并提取关键结果

```python
# 拟合 PCA
pca = PCA(n_components=6)   # 先保留全部，再根据碎石图决定 k
pca.fit(X_sc)

# 解释方差比
evr = pca.explained_variance_ratio_
print('各主成分解释方差比：')
for i, r in enumerate(evr):
    print(f'  PC{i+1}: {r:.1%}')
print(f'前 2 个 PC 累计: {evr[:2].sum():.1%}')

# 因子载荷矩阵：shape = (n_vars, n_components)
loadings = pca.components_.T
df_load = pd.DataFrame(
    loadings,
    index=stock_names,
    columns=[f'PC{i+1}' for i in range(6)]
)
print('\n因子载荷矩阵（前 2 个 PC）：')
print(df_load.iloc[:, :2].round(3))

# 主成分得分：shape = (n_samples, n_components)
scores = pca.transform(X_sc)
print(f'\n得分矩阵形状: {scores.shape}')
print(f'PC1 与 PC2 得分的相关系数: '
      f'{np.corrcoef(scores[:,0], scores[:,1])[0,1]:.8f}')
```

### 碎石图

```python
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8, 5))

pcs = range(1, 7)
ax1.bar(pcs, evr * 100, color='#1A3A6B', alpha=0.75, width=0.5)
ax1.set_xlabel('主成分编号')
ax1.set_ylabel('解释方差比 (%)')

ax2 = ax1.twinx()
ax2.plot(pcs, np.cumsum(evr) * 100, 'o-', color='#C8900A', lw=2)
ax2.set_ylabel('累计解释方差 (%)')
ax2.set_ylim(0, 108)

plt.title('碎石图')
plt.tight_layout()
plt.show()
```

### 主成分回归（PCR）

```python
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline

# 读取宏观数据
df_M = pd.read_csv('./data/pca_macro.csv')
macro_cols = [c for c in df_M.columns if c != 'GDP_growth']
X_M = df_M[macro_cols].values
y_M = df_M['GDP_growth'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 遍历不同 k，比较 CV MSE
cv_mse = {}
for k in range(1, 9):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=k)),
        ('reg',    LinearRegression()),
    ])
    scores = -cross_val_score(
        pipe, X_M, y_M, cv=kf,
        scoring='neg_mean_squared_error'
    )
    cv_mse[k] = scores.mean()

best_k = min(cv_mse, key=cv_mse.get)
print(f'最优主成分数: k={best_k}')
print(f'对应 CV MSE: {cv_mse[best_k]:.4f}')

# 用最优 k 在全训练集上拟合
pipe_final = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=best_k)),
    ('reg',    LinearRegression()),
])
pipe_final.fit(X_M, y_M)
print(f'训练集 R²: {pipe_final.score(X_M, y_M):.4f}')
```

---

## 小结

PCA 是统计学习中最基础、最广泛使用的无监督方法之一。本章建立的核心知识链条是：

$$
\underbrace{\text{方差最大化}}_{ \mathbf{v}_1 = \arg\max \mathbf{v}^\top \mathbf{S} \mathbf{v}}
\xrightarrow{\text{特征分解}}
\underbrace{\text{载荷 + 得分}}_{\mathbf{Z} = \mathbf{X}\mathbf{V}}
\xrightarrow{\text{碎石图}}
\underbrace{\text{选 } k \text{ 个主成分}}_{ \text{EVR 或 CV}}
\xrightarrow{\text{下游任务}}
\underbrace{\text{PCR / 可视化 / 聚类}}_{ \text{降维的应用}}
$$

**五个关键结论：**

1. PCA 是正交坐标变换，不增加也不减少信息，只是换了一个坐标系，   使方差在前几个轴上最大化

2. 第 $k$ 个主成分的载荷向量是样本协方差矩阵（标准化后为相关矩阵）   第 $k$ 大特征值对应的特征向量；解释方差比等于该特征值除以所有特征值之和

3. 碎石图的「肘部」是选择主成分数量的直观依据；   以预测为目标时，应用交叉验证选择 $k$

4. 双标图同时展示样本和变量，箭头夹角反映变量间的相关性，   是理解主成分经济含义的最直接工具

5. 主成分回归（PCR）以互不相关的主成分得分替代原始高度相关的变量，   是缓解多重共线性的有效方案；使用 `Pipeline` 可正确处理标准化的数据泄露问题

---

## 附录 A　特征分解：PCA 最优解的数学推导

**问题**：在 $\|\mathbf{v}\|=1$ 的约束下，最大化 $\mathbf{v}^\top \mathbf{S} \mathbf{v}$。

构造拉格朗日函数：

$$
\mathcal{L}(\mathbf{v}, \lambda)
= \mathbf{v}^\top \mathbf{S} \mathbf{v} - \lambda(\mathbf{v}^\top \mathbf{v} - 1)
$$

对 $\mathbf{v}$ 求导并令其为零：

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{v}} = 2\mathbf{S}\mathbf{v} - 2\lambda\mathbf{v} = 0
\implies \mathbf{S}\mathbf{v} = \lambda\mathbf{v}
$$

这正是矩阵特征方程。最优的 $\mathbf{v}$ 是 $\mathbf{S}$ 的特征向量，对应的目标函数值为：

$$
\mathbf{v}^\top \mathbf{S} \mathbf{v} = \mathbf{v}^\top (\lambda \mathbf{v})
= \lambda \|\mathbf{v}\|^2 = \lambda
$$

因此，使目标函数最大的特征向量是**最大特征值** $\lambda_1$ 对应的特征向量，这就是第一主成分方向。

对后续主成分，在已有特征向量的正交补空间中重复上述推导，依次得到特征值从大到小排列的特征向量序列。由于 $\mathbf{S}$ 是半正定对称矩阵，所有特征值非负，所有特征向量互相正交。

## 附录 B　奇异值分解（SVD）与 PCA 的等价性

实践中，sklearn 的 PCA 实现不是直接对协方差矩阵做特征分解，而是对数据矩阵 $\mathbf{X}$（已中心化）做**奇异值分解**（SVD）：

$$
\mathbf{X} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^\top \tag{B.1}
$$

其中 $\mathbf{U} \in \mathbb{R}^{n \times p}$ 是左奇异向量矩阵，$\boldsymbol{\Sigma}$ 是奇异值对角矩阵，$\mathbf{V} \in \mathbb{R}^{p \times p}$ 是右奇异向量矩阵。

SVD 与 PCA 的对应关系为：

- $\mathbf{V}$ 的列 = 主成分方向（载荷向量）
- $\mathbf{U}\boldsymbol{\Sigma}$ = 主成分得分矩阵（$\mathbf{Z} = \mathbf{XV}$）
- 奇异值 $\sigma_k$ 与特征值的关系：$\lambda_k = \sigma_k^2 / (n-1)$

SVD 算法比直接对协方差矩阵做特征分解更数值稳定，尤其是当 $n \ll p$（样本少、变量多）时优势显著。

另外，当 $n \ll p$ 时，`PCA(n_components=k)` 内部使用截断 SVD（Truncated SVD），只计算前 $k$ 个奇异值/向量，复杂度从 $O(p^3)$ 降低到 $O(npk)$，这对金融中常见的高维数据（如大截面股票因子）非常重要。

## 参考文献

- James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning: with Applications in Python* (2nd ed.). Springer. Chapter 12. [Link](https://www.statlearning.com/)
- Jolliffe, I. T. (2002). *Principal Component Analysis* (2nd ed.). Springer. [Link](https://doi.org/10.1007/b98835)
- Bai, J., & Ng, S. (2002). Determining the number of factors in approximate factor models. *Econometrica*, 70(1), 191–221. [Link](https://doi.org/10.1111/1468-0262.00273)
- Connor, G., & Korajczyk, R. A. (1988). Risk and return in an equilibrium APT. *Journal of Financial Economics*, 21(2), 255–289. [Link](https://doi.org/10.1016/0304-405X(88)90062-1)